In [1]:
import sys
import os
import json
import urllib3
import requests
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)


Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [ ]:
import pandas as pd
from datetime import datetime

# Define time period
start = "2025-01-01T00:00:00Z"  # adjust as needed

# List of owners to pull from
list_of_owners = ['HTOC Org', 'HTOC Comm', 'HTOC legacy data']
final_results = []
# Enrich a specific indicator
summary = '172.240.108.68'
enrich_type = "Shodan"

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        # Build TQL query string
        tql = f'ownerName EQ "{owner}" and lastObserved >= "{start}" and (indicatorActive EQ true OR indicatorActive EQ false) and summary EQ "{summary}"'

        # Set up the GET request for indicators
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators/?tql={tql}&fields=threatAssess,associatedGroups&resultStart=0&resultLimit=10000')

        # Send the GET request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

        # Enrich the specific indicator using POST
        print(f"\nEnriching indicator: {summary} with type: {enrich_type}")
        ro.set_http_method('POST')
        ro.set_request_uri(f'/v3/indicators/{summary}/enrich?type={enrich_type}')
        enrich_response = tc.api_request(ro)

        # Parse enrichment response
        if enrich_response.headers.get('content-type') == 'application/json':
            enriched_data = enrich_response.json()
            print(f"Enriched data for indicator {summary}:")
        else:
            print(f"Unexpected response format during enrichment: {enrich_response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query or enrich indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    observed_src = pd.json_normalize(final_results, record_path='data')
    observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
    print(f"\nRetrieved {len(observed_src)} observed indicators.")
else:
    print("\nNo indicators retrieved.")
    
observed_src



Querying owner: HTOC Org

Enriching indicator: 172.240.108.68 with type: Shodan
Enriched data for indicator 172.240.108.68:

Querying owner: HTOC Comm

Enriching indicator: 172.240.108.68 with type: Shodan
Enriched data for indicator 172.240.108.68:

Querying owner: HTOC legacy data

Enriching indicator: 172.240.108.68 with type: Shodan
Enriched data for indicator 172.240.108.68:

Retrieved 1 observed indicators.


In [11]:
print(json.dumps(enriched_data, indent=4))


{
    "data": {
        "id": 4685359,
        "dateAdded": "2024-05-29T18:34:07Z",
        "ownerId": 9,
        "ownerName": "HTOC Org",
        "webLink": "https://hvs.threatconnect.com/#/details/indicators/4685359",
        "type": "Address",
        "lastModified": "2025-04-21T13:18:40Z",
        "rating": 4.0,
        "confidence": 78,
        "description": "On April 23, 2024, VA SEIM alerted to malicious beaconing communications from a VA host to domain <mavink[.]com\u201d> (IP: 172.240.108[.]68) and then to \u201ccertifiedblob[.]com\u201d to retrieve a known malicious JavaScript file \u201cinvoke.js\u201d. The obfuscated JavaScript code referenced the following domains: <https://proftrafficcounter.com/stats> and <https://scratchconsonant.com/watch>.\nA security researcher submitted a variation of the \u201cinvoke.js\u201d file (originally hosted by the known malicious domain <topcre",
        "summary": "172.240.108.68",
        "privateFlag": false,
        "active": true,
  